# Role declarations accept role choices and reject removed conditions.

Run the cells in order. The matching [script](24_role_declarations_and_migration.py) has the same declarations, calls and assertions.

See the [guide](../../docs/how-to/roles-and-object-rules.md).

## Declarations

Import the public role and verdict APIs and declare two independent roles.

In [1]:
"""Role declarations accept role choices and reject removed conditions.
Run: .venv/bin/python examples/step_03_authorization_and_roles/24_role_declarations_and_migration.py
"""

from __future__ import annotations

from pydantic import ValidationError

from aoa.action_machine.auth import ApplicationRole, GuestRole
from aoa.action_machine.exceptions import RoleSpecTypeError
from aoa.action_machine.intents.access_control import Gate, Refused
from aoa.action_machine.intents.check_roles import check_roles

# %% Setup


class ManagerRole(ApplicationRole):
    """Permit order management."""

    name = "manager"
    description = "Manage orders"


class AuditorRole(ApplicationRole):
    """Permit independent reviews."""

    name = "auditor"
    description = "Review orders"


## Accepted role forms

Legal roles, alternatives, collections and repeated roles remain accepted.

In [2]:
check_roles(ManagerRole)
check_roles(ManagerRole, AuditorRole)
check_roles((ManagerRole, AuditorRole))
check_roles([ManagerRole, AuditorRole])
check_roles(ManagerRole, ManagerRole)
check_roles(GuestRole, ManagerRole)
print("accepted: one role, positional roles, tuple, list, duplicates, guest alternative")


accepted: one role, positional roles, tuple, list, duplicates, guest alternative


## Removed condition keywords

Each old condition keyword must fail immediately with migration guidance.

In [3]:
for keyword in ("guard", "guard_reason", "when"):
    try:
        check_roles(ManagerRole, **{keyword: "old condition"})
    except RoleSpecTypeError as exc:
        expected = (
            f"@check_roles declares roles only; {keyword!r} belonged to the removed condition layer. "
            "Move the condition into the operation's @access_decide method and return "
            'Refused("REASON") or FORBIDDEN_OBJECT for the branch that used to refuse.'
        )
        assert str(exc) == expected
        print(type(exc).__name__ + ":", str(exc))
    else:
        raise AssertionError(f"The removed keyword {keyword} must be refused.")


RoleSpecTypeError: @check_roles declares roles only; 'guard' belonged to the removed condition layer. Move the condition into the operation's @access_decide method and return Refused("REASON") or FORBIDDEN_OBJECT for the branch that used to refuse.
RoleSpecTypeError: @check_roles declares roles only; 'guard_reason' belonged to the removed condition layer. Move the condition into the operation's @access_decide method and return Refused("REASON") or FORBIDDEN_OBJECT for the branch that used to refuse.
RoleSpecTypeError: @check_roles declares roles only; 'when' belonged to the removed condition layer. Move the condition into the operation's @access_decide method and return Refused("REASON") or FORBIDDEN_OBJECT for the branch that used to refuse.


## Removed grant imports

Both the former factory and its type must fail at import time.

In [4]:
try:
    from aoa.action_machine.intents.check_roles import grant
except ImportError as exc:
    assert "cannot import name 'grant'" in str(exc)
    print(type(exc).__name__ + ":", str(exc))
else:
    raise AssertionError("The removed grant import must fail.")
try:
    from aoa.action_machine.intents.check_roles import Grant
except ImportError as exc:
    assert "cannot import name 'Grant'" in str(exc)
    print(type(exc).__name__ + ":", str(exc))
else:
    raise AssertionError("The removed Grant import must fail.")


ImportError: cannot import name 'grant' from 'aoa.action_machine.intents.check_roles' (/Users/bystrovmaxim/PythonDev/aoa/packages/aoa-action-machine/src/aoa/action_machine/intents/check_roles/__init__.py)
ImportError: cannot import name 'Grant' from 'aoa.action_machine.intents.check_roles' (/Users/bystrovmaxim/PythonDev/aoa/packages/aoa-action-machine/src/aoa/action_machine/intents/check_roles/__init__.py)


## Accepted role forms

Legal roles, alternatives, collections and repeated roles remain accepted.

In [5]:
import aoa.action_machine.exceptions as exceptions


## Removed condition exceptions

The former synchronous-condition exception types are no longer published.

In [6]:
assert not hasattr(exceptions, "AccessConditionAsyncError")
assert not hasattr(exceptions, "ConditionReasonError")


## Gate vocabulary and payloads

Check all three current values, reject old member names and payload values, and retain a valid default refusal.

In [7]:
assert [gate.value for gate in Gate] == ["AUTH_COORDINATOR", "CHECK_ROLES", "ACCESS_DECIDE"]
print("gates:", [gate.value for gate in Gate])
for word in ("WHEN", "GUARD"):
    try:
        getattr(Gate, word)
    except AttributeError as exc:
        assert str(exc) == f"type object 'Gate' has no attribute {word!r}"
        print(type(exc).__name__ + ":", str(exc))
    else:
        raise AssertionError("Removed gate members must be absent.")
    try:
        Gate(word)
    except ValueError as exc:
        assert str(exc) == f"{word!r} is not a valid Gate"
        print(type(exc).__name__ + ":", str(exc))
    else:
        raise AssertionError("Removed gate words must fail enum conversion.")
    try:
        Refused.model_validate({"gate": word})
    except ValidationError as exc:
        error = exc.errors(include_url=False)[0]
        assert error["loc"] == ("gate",) and error["type"] == "enum"
        assert error["msg"] == "Input should be 'AUTH_COORDINATOR', 'CHECK_ROLES' or 'ACCESS_DECIDE'"
        print(type(exc).__name__ + ":", str(exc))
    else:
        raise AssertionError("Removed gate words must fail payload validation.")
assert Refused().gate is Gate.ACCESS_DECIDE
print("default refusal:", Refused().model_dump(mode="json"))


gates: ['AUTH_COORDINATOR', 'CHECK_ROLES', 'ACCESS_DECIDE']
AttributeError: type object 'Gate' has no attribute 'WHEN'
ValueError: 'WHEN' is not a valid Gate
ValidationError: 1 validation error for Refused
gate
  Input should be 'AUTH_COORDINATOR', 'CHECK_ROLES' or 'ACCESS_DECIDE' [type=enum, input_value='WHEN', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/enum
AttributeError: type object 'Gate' has no attribute 'GUARD'
ValueError: 'GUARD' is not a valid Gate
ValidationError: 1 validation error for Refused
gate
  Input should be 'AUTH_COORDINATOR', 'CHECK_ROLES' or 'ACCESS_DECIDE' [type=enum, input_value='GUARD', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/enum
default refusal: {'kind': 'refused', 'gate': 'ACCESS_DECIDE', 'reason': None}
